# GAS Raw Requests Usage

This notebook demonstrates the GAS Server HTTP interface directly with `requests`. It uses only plain HTTP requests so you can see the raw operation URLs, request bodies, streaming events, task polling, and cancellation behavior.

Covered operations:

1. `GetCapabilities`
2. `DescribeAgent`
3. `GetAgentStatus`
4. `ExecuteTask` with `sync` mode
5. `ExecuteTask` with `async` mode
6. `GetTaskStatus`
7. `GetTaskResult`
8. `ExecuteTask` with `stream` mode
9. `CancelTask`

## Imports

Only plain `requests` calls are used. A few local helper functions format stream events and task summaries for readability.

In [ ]:
import json
import time
from urllib.parse import urljoin, urlparse

import requests
from IPython.display import Image, display



## User Settings

Start the GAS server first, then adjust these values if needed.

In [ ]:
server_url = "http://127.0.0.1:4042"
retrieval_agent_id = "geospatial_data_retrieval_agent"
mapping_agent_id = "mapping_agent"
status_agent_id = retrieval_agent_id

sync_task_text = "Download Pennsylvania county boundaries from Census Bureau."
async_task_text = "Download county-level 2021 population for the contiguous United States."
stream_task_text = "Create a county-level choropleth map of the 2021 population using a quantile classification scheme with 5 classes. Use Lambert Conformal Conic map projection."

openai_api_key = "YOUR_OPENAI_API_KEY"
us_census_demography_key = "YOUR_CENSUS_API_KEY"



## Small Raw-Request Helpers

These helpers keep the notebook readable. They still use plain `requests` and the operation URLs advertised by `GetCapabilities`.

In [ ]:
def pretty(payload):
    print(json.dumps(payload, indent=2))



def format_display_value(value):
    if value in (None, "", [], {}):
        return "-"
    if isinstance(value, int):
        return f"{value:,}"
    if isinstance(value, float):
        return f"{value:,.2f}"
    return str(value)


def format_duration_seconds(value):
    if value in (None, ""):
        return "-"
    try:
        return f"{float(value):.2f}s"
    except Exception:
        return str(value)


def stream_event_time(event):
    timestamp = event.get("timestamp")
    if not timestamp:
        return "--:--:--"
    try:
        from datetime import datetime
        return datetime.fromisoformat(str(timestamp).replace("Z", "+00:00")).astimezone().strftime("%H:%M:%S")
    except Exception:
        return str(timestamp)


def format_stream_message(event, agent_name=None):
    message = str(event.get("message") or event.get("status") or "")
    event_type = event.get("event")

    if event_type != "progress":
        return message

    if message.startswith("The user wants help from "):
        return "I received your request."

    if " is still working. Long LLM calls, code execution, or geospatial file processing can take a little while." in message:
        return "I am still working. Long LLM calls, code execution, or geospatial file processing can take a little while."

    return message


def print_stream_event(event, agent_name=None):
    time_text = stream_event_time(event)
    event_type = event.get("event") or event.get("kind") or "event"

    if event_type == "task_result":
        payload = event.get("payload") if isinstance(event.get("payload"), dict) else {}
        task = payload.get("task") if isinstance(payload.get("task"), dict) else {}
        task_id = task.get("id") or event.get("task_id")
        print(f"[{time_text}] task_result: final task received {task_id or ''}".rstrip())
        return

    label = agent_name if event_type == "progress" and agent_name else event_type
    message = format_stream_message(event, agent_name=agent_name)
    print(f"[{time_text}] {label}: {message}".rstrip())


def print_task_summary(task_result):
    task = task_result.get("task") if isinstance(task_result.get("task"), dict) else {}
    agent = task_result.get("agent") if isinstance(task_result.get("agent"), dict) else {}
    outputs = task_result.get("outputs") if isinstance(task_result.get("outputs"), dict) else {}
    execution = task_result.get("execution") if isinstance(task_result.get("execution"), dict) else {}
    provenance = task_result.get("provenance") if isinstance(task_result.get("provenance"), dict) else {}
    diagnostics = task_result.get("diagnostics") if isinstance(task_result.get("diagnostics"), dict) else {}
    token_usage = provenance.get("token_usage") if isinstance(provenance.get("token_usage"), dict) else {}
    artifacts = outputs.get("artifacts") if isinstance(outputs.get("artifacts"), list) else []

    input_tokens = token_usage.get("input_tokens")
    output_tokens = token_usage.get("output_tokens")
    total_tokens = token_usage.get("total_tokens")
    if total_tokens in (None, "") and isinstance(input_tokens, int) and isinstance(output_tokens, int):
        total_tokens = input_tokens + output_tokens

    print("\n" + "=" * 72)
    print("GAS Task Summary")
    print("=" * 72)
    print(f"Task         : {format_display_value(task.get('id'))}")
    print(f"Status       : {format_display_value(task.get('status'))}")
    print(f"Agent        : {format_display_value(agent.get('name') or agent.get('id'))}")
    print(f"Version      : {format_display_value(agent.get('version'))}")
    print(f"Model        : {format_display_value(agent.get('model'))}")
    print(f"Duration     : {format_duration_seconds(execution.get('duration_seconds'))}")
    print(f"Iterations   : {format_display_value(execution.get('iterations'))}")

    print("\nUsage")
    print("-----")
    print(f"LLM calls    : {format_display_value(provenance.get('llm_calls'))}")
    print(f"Tool calls   : {format_display_value(provenance.get('tool_calls'))}")
    print(f"Input tokens : {format_display_value(input_tokens)}")
    print(f"Output tokens: {format_display_value(output_tokens)}")
    print(f"Total tokens : {format_display_value(total_tokens)}")

    print("\nOutputs")
    print("-------")
    print(f"Summary      : {format_display_value(outputs.get('summary'))}")
    print(f"Artifacts    : {len(artifacts)}")

    for index, artifact in enumerate(artifacts, start=1):
        if not isinstance(artifact, dict):
            continue
        name = artifact.get("filename") or artifact.get("name") or f"artifact_{index}"
        artifact_type = artifact.get("type")
        artifact_format = artifact.get("format") or artifact.get("mime_type")
        size_bytes = artifact.get("size_bytes")
        print(f"  {index}. {name}")
        print(
            "     "
            f"type={format_display_value(artifact_type)} "
            f"format={format_display_value(artifact_format)} "
            f"size={format_display_value(size_bytes)} bytes"
        )
        if artifact.get("url"):
            print(f"     url={artifact['url']}")

    print("\nDiagnostics")
    print("-----------")
    print(f"Has error    : {format_display_value(diagnostics.get('has_error'))}")
    if diagnostics.get("error"):
        print(f"Error        : {diagnostics.get('error')}")
    warnings = diagnostics.get("warnings") if isinstance(diagnostics.get("warnings"), list) else []
    if warnings:
        print("Warnings     :")
        for warning in warnings:
            print(f"  - {warning}")
    else:
        print("Warnings     : -")
    print("=" * 72)


def absolute_url(url):
    parsed = urlparse(url)
    if parsed.scheme and parsed.netloc:
        path_and_query = parsed.path
        if parsed.query:
            path_and_query += f"?{parsed.query}"
        return urljoin(server_url, path_and_query)
    if url.startswith("/"):
        return urljoin(server_url, url)
    return url


def get_json(url, *, timeout=30):
    print("GET", url)
    response = requests.get(url, timeout=timeout)
    print("HTTP status:", response.status_code)
    response.raise_for_status()
    return response.json()


def post_json(url, payload, *, timeout=30, stream=False):
    print("POST", url)
    response = requests.post(url, json=payload, timeout=timeout, stream=stream)
    print("HTTP status:", response.status_code)
    response.raise_for_status()
    return response


def ensure_operations():
    global capabilities, operations

    if "operations" in globals() and operations:
        return operations

    capabilities_url = f"{server_url}/?SERVICE=GAS&VERSION=1.0.0&REQUEST=GetCapabilities"
    capabilities = get_json(capabilities_url, timeout=10)
    operations = {
        operation.get("operation_id"): operation
        for operation in capabilities.get("operations", [])
    }
    return operations


def operation_url(operation_id, agent_id=None, task_id=None):
    operation_map = ensure_operations()
    operation = operation_map[operation_id]
    url = operation["url"]
    if agent_id is not None:
        url = url.replace("{agent_id}", agent_id)
    if task_id is not None:
        url = url.replace("{task_id}", task_id)
    return absolute_url(url)


def build_execute_task_payload(
    instructions,
    *,
    mode="sync",
    input_datasets=None,
    artifact_delivery="URL",
    parameters=None,
    credentials=None,
    metadata=None,
):
    payload = {
        "task": {
            "instructions": instructions,
            "mode": mode,
        },
        "outputs": {
            "artifact_delivery": artifact_delivery,
        },
    }

    if input_datasets is not None:
        payload["inputs"] = {
            "input_datasets": input_datasets if isinstance(input_datasets, list) else [input_datasets]
        }

    if parameters:
        payload["parameters"] = parameters

    if credentials:
        payload["credentials"] = credentials

    if metadata:
        payload["metadata"] = metadata

    return payload


def first_artifact_url(task_result, preferred_extensions=None):
    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    preferred_extensions = preferred_extensions or []

    for extension in preferred_extensions:
        for artifact in artifacts:
            url = artifact.get("url")
            filename = artifact.get("filename") or artifact.get("name") or url or ""
            if url and str(filename).lower().endswith(extension.lower()):
                return absolute_url(url)

    for artifact in artifacts:
        if artifact.get("url"):
            return absolute_url(artifact["url"])

    raise RuntimeError("The task returned no artifact URL.")


def poll_task(agent_id, task_id, *, interval_seconds=5, timeout_seconds=600):
    status_url = operation_url("get_task_status", agent_id=agent_id, task_id=task_id)
    result_url = operation_url("get_task_result", agent_id=agent_id, task_id=task_id)
    terminal_statuses = {"successful", "failed", "canceled", "rejected"}
    started = time.time()

    while True:
        task_status = get_json(status_url, timeout=30)
        status = task_status.get("task", {}).get("status")
        print("Task status:", status)

        if status in terminal_statuses:
            break

        if time.time() - started > timeout_seconds:
            raise TimeoutError(f"Task {task_id} did not finish within {timeout_seconds} seconds.")

        time.sleep(interval_seconds)

    return get_json(result_url, timeout=30)

## 1. GetCapabilities

`GetCapabilities` is the server-level discovery document. It lists shared GAS operations and published agents.

In [ ]:
capabilities_url = f"{server_url}/?SERVICE=GAS&VERSION=1.0.0&REQUEST=GetCapabilities"
capabilities = get_json(capabilities_url, timeout=10)

print("GAS version:", capabilities.get("version"))
print("Number of operations:", len(capabilities.get("operations", [])))
print("Number of agents:", len(capabilities.get("agents", [])))

In [ ]:
pretty(capabilities)

In [ ]:
operations = ensure_operations()

published_agents = [
    agent.get("agent_id")
    for agent in capabilities.get("agents", [])
]

print("Operations:", ", ".join(operations))
print("Agents:", ", ".join(published_agents))


## 2. DescribeAgent

`DescribeAgent` is the agent-level discovery document. It describes the selected agent profile, skills, task request details, credentials, parameters, and response schema.

In [ ]:
describe_url = f"{server_url}/?SERVICE=GAS&VERSION=1.0.0&REQUEST=DescribeAgent&agent_id={retrieval_agent_id}"
retrieval_description = get_json(describe_url, timeout=30)

profile = retrieval_description.get("profile", {})
execute_task = retrieval_description.get("execute_task", {})

print("Agent ID:", profile.get("agent_id"))
print("Agent name:", profile.get("name"))
print("ExecuteTask modes:", ", ".join(execute_task.get("modes", [])))
print("Requires input datasets:", execute_task.get("inputs", {}).get("input_datasets", {}).get("required"))

In [ ]:
pretty(retrieval_description)

## 3. GetAgentStatus

`GetAgentStatus` checks whether a published agent service is available.

In [ ]:
status_url = operation_url("get_agent_status", agent_id=status_agent_id)
agent_status = get_json(status_url, timeout=30)
pretty(agent_status)

## 4. ExecuteTask, Sync Mode

`sync` mode waits for the task to finish and returns the standard GAS task result directly in the HTTP response.

In [ ]:
sync_payload = build_execute_task_payload(
    sync_task_text,
    mode="sync",
    artifact_delivery="URL",
    credentials={
        "OPENAI_API_KEY": openai_api_key,
    },
    metadata={
        "client_id": "gas_raw_requests_usage.ipynb",
        "example": "sync_mode",
    },
)

pretty(sync_payload)

In [ ]:
sync_url = operation_url("execute_task", agent_id=retrieval_agent_id)
sync_response = post_json(sync_url, sync_payload, timeout=600)
sync_result = sync_response.json()

print_task_summary(sync_result)

In [ ]:
pretty(sync_result)

## 5. ExecuteTask, Async Mode

`async` mode returns quickly with a task ID. Use `GetTaskStatus` and `GetTaskResult` to retrieve the final response later.

In [ ]:
async_payload = build_execute_task_payload(
    async_task_text,
    mode="async",
    artifact_delivery="URL",
    credentials={
        "OPENAI_API_KEY": openai_api_key,
        "source_credentials": {
            "US_Census_demography": {
                "key": us_census_demography_key,
            }
        },
    },
    metadata={
        "client_id": "gas_raw_requests_usage.ipynb",
        "example": "async_mode",
    },
)

pretty(async_payload)

In [ ]:
async_url = operation_url("execute_task", agent_id=retrieval_agent_id)
async_response = post_json(async_url, async_payload, timeout=30)
async_submission = async_response.json()
pretty(async_submission)

async_task_id = async_submission["task"]["id"]
print("Task ID:", async_task_id)

## 6. GetTaskStatus and 7. GetTaskResult

Poll the status endpoint until the async task is terminal, then request the final task result.

In [ ]:
async_result = poll_task(retrieval_agent_id, async_task_id, interval_seconds=5, timeout_seconds=900)
print_task_summary(async_result)

In [ ]:
pretty(async_result)

## 8. ExecuteTask, Stream Mode

`stream` mode keeps the HTTP connection open and sends newline-delimited JSON events. The final event is `task_result`, whose payload is the same standard GAS response document returned by sync/async result retrieval.

This example maps the dataset returned by the async retrieval task.

In [ ]:
dataset_url = first_artifact_url(async_result, preferred_extensions=[".geojson", ".json", ".gpkg", ".zip"])
dataset_url

In [ ]:
stream_payload = build_execute_task_payload(
    stream_task_text,
    mode="stream",
    input_datasets=[dataset_url],
    artifact_delivery="URL",
    credentials={
        "OPENAI_API_KEY": openai_api_key,
    },
    metadata={
        "client_id": "gas_raw_requests_usage.ipynb",
        "example": "stream_mode",
    },
)

pretty(stream_payload)

In [ ]:
stream_url = operation_url("execute_task", agent_id=mapping_agent_id)
stream_response = post_json(stream_url, stream_payload, timeout=900, stream=True)

stream_result = None
stream_task_id = None

for line in stream_response.iter_lines(decode_unicode=True):
    if not line:
        continue

    event = json.loads(line)
    print_stream_event(event, agent_name="Mapping Agent")

    if event.get("task_id"):
        stream_task_id = event.get("task_id")

    if event.get("event") == "task_result":
        stream_result = event.get("payload")
        if isinstance(stream_result, dict):
            stream_task_id = stream_result.get("task", {}).get("id") or stream_task_id
        break

print("Stream task ID:", stream_task_id)

In [ ]:
print_task_summary(stream_result)
pretty(stream_result)

In [ ]:
stream_artifacts = stream_result.get("outputs", {}).get("artifacts", []) if isinstance(stream_result, dict) else []

for artifact in stream_artifacts:
    print("-", artifact.get("filename") or artifact.get("name"))
    artifact_url = artifact.get("url")
    if artifact_url:
        display_url = absolute_url(artifact_url)
        print("  URL:", display_url)
        filename = artifact.get("filename") or artifact.get("name") or display_url
        if str(filename).lower().endswith(".png"):
            display(Image(url=display_url))

## 9. CancelTask

`CancelTask` asks the server to cancel an active async task. If the task has already finished, the server should return a conflict response because terminal tasks cannot be canceled.

This cell uses a deliberately long-running mapping request. If the task finishes before the cancel request reaches it, you may see a `409` response instead of a cancellation result. That is still useful behavior to understand.

In [ ]:
cancel_demo_payload = build_execute_task_payload(
    "Create a detailed map with multiple classification attempts and a careful visual summary. This task is only for demonstrating CancelTask.",
    mode="async",
    input_datasets=[dataset_url],
    artifact_delivery="URL",
    credentials={
        "OPENAI_API_KEY": openai_api_key,
    },
    metadata={
        "client_id": "gas_raw_requests_usage.ipynb",
        "example": "cancel_task",
    },
)

cancel_demo_response = post_json(operation_url("execute_task", agent_id=mapping_agent_id), cancel_demo_payload, timeout=30)
cancel_demo_submission = cancel_demo_response.json()
cancel_demo_task_id = cancel_demo_submission["task"]["id"]
print("Cancel-demo task ID:", cancel_demo_task_id)
pretty(cancel_demo_submission)

In [ ]:
cancel_url = operation_url("cancel_task", agent_id=mapping_agent_id, task_id=cancel_demo_task_id)
print("POST", cancel_url)

cancel_response = requests.post(cancel_url, timeout=30)
print("HTTP status:", cancel_response.status_code)

try:
    cancel_payload = cancel_response.json()
except Exception:
    cancel_payload = {"raw_text": cancel_response.text}

pretty(cancel_payload)

In [ ]:
# Confirm the final status after cancellation or after a too-late cancel attempt.
cancel_demo_status_url = operation_url("get_task_status", agent_id=mapping_agent_id, task_id=cancel_demo_task_id)
cancel_demo_status = get_json(cancel_demo_status_url, timeout=30)
pretty(cancel_demo_status)